In [ ]:
# Cell 1 : Imports and runtime config
import asyncio
import time

import matplotlib.pyplot as plt
import numpy as np
import requests
import websockets

API_BASE = "http://127.0.0.1:8080"
WS_BASE = "ws://127.0.0.1:8080"

CENTER_FREQ_HZ = 915_000_000
SAMPLE_RATE_SPS = 2_000_000
LNA_GAIN_DB = 40
VGA_GAIN_DB = 56
AMP_ENABLE = True
TARGET_IQ_SAMPLES = 262_144
FRAMES_PER_FREQ = 10

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True

In [ ]:
# Cell 2 : HTTP helper functions
def api_get(path: str):
    r = requests.get(f"{API_BASE}{path}", timeout=10)
    r.raise_for_status()
    return r.json()

def api_post(path: str, payload: dict):
    r = requests.post(f"{API_BASE}{path}", json=payload, timeout=20)
    r.raise_for_status()
    return r.json()

In [ ]:
# Cell 3 : Health check and device discovery
health = api_get('/health')
devices = api_get('/devices')

print('Health:', health)
print('Devices:')
for d in devices:
    print(' -', d['id'], d['label'], f"[{d['freq_min_hz']}..{d['freq_max_hz']} Hz]")

if not devices:
    raise RuntimeError('No SDR devices returned by /devices')

# Prefer HackRF if present, else first device (e.g. mock backend).
device = next((d for d in devices if d['driver'] == 'hackrf'), devices[0])
DEVICE_ID = device['id']
print('Using device:', DEVICE_ID)

In [ ]:
# Cell 4 : Define frequency targets and stream config
FREQ_TARGETS_HZ = [
    98_100_000,   # FM broadcast
    751_000_000,  # LTE band region
    2_402_000_000 # WiFi/Bluetooth ISM
]

STREAM_PARAMS = {
    'device_id': DEVICE_ID,
    'sample_rate_sps': SAMPLE_RATE_SPS,
    'lna_gain_db': LNA_GAIN_DB,
    'vga_gain_db': VGA_GAIN_DB,
    'amp_enable': AMP_ENABLE
}

print('Target centers (MHz):', [f / 1e6 for f in FREQ_TARGETS_HZ])


In [ ]:
# Cell 5 : Receive IQ bytes over WebSocket
async def recv_iq_bytes(stream_id: str, target_samples: int):
    # HackRF stream is interleaved int8 IQ, so 2 bytes per complex sample.
    target_bytes = target_samples * 2
    data = bytearray()
    uri = f"{WS_BASE}/ws/iq/{stream_id}"

    async with websockets.connect(uri, max_size=None, ping_interval=20, ping_timeout=20) as ws:
        while len(data) < target_bytes:
            chunk = await asyncio.wait_for(ws.recv(), timeout=10)
            if isinstance(chunk, str):
                chunk = chunk.encode('utf-8')
            data.extend(chunk)

    return bytes(data[:target_bytes])


In [ ]:
# Cell 6 : Capture 10 frames and plot average + persistence (max)
def iq_bytes_to_complex(iq_raw: bytes):
    iq_i8 = np.frombuffer(iq_raw, dtype=np.int8)
    iq = iq_i8[0::2].astype(np.float32) + 1j * iq_i8[1::2].astype(np.float32)
    return iq - np.mean(iq)

def frame_psd_db(iq, center_freq_hz: int, sample_rate_sps: int):
    n_fft = min(len(iq), 131072)
    window = np.hanning(n_fft).astype(np.float32)
    spec = np.fft.fftshift(np.fft.fft(iq[:n_fft] * window))
    psd_db = 20 * np.log10(np.abs(spec) + 1e-12)
    freq_axis_hz = np.linspace(-sample_rate_sps / 2, sample_rate_sps / 2, n_fft, endpoint=False) + center_freq_hz
    return freq_axis_hz, psd_db

captures = []
for freq_hz in FREQ_TARGETS_HZ:
    payload = dict(STREAM_PARAMS)
    payload['center_freq_hz'] = freq_hz

    stream_resp = api_post('/streams/start', payload)
    stream_id = stream_resp['stream_id']
    print(f'Started stream for {freq_hz/1e6:.3f} MHz -> {stream_id}')

    frame_psd = []
    freq_axis_hz = None
    for i in range(FRAMES_PER_FREQ):
        iq_raw = await recv_iq_bytes(stream_id, TARGET_IQ_SAMPLES)
        iq = iq_bytes_to_complex(iq_raw)
        freq_axis_hz, psd_db = frame_psd_db(iq, freq_hz, SAMPLE_RATE_SPS)
        frame_psd.append(psd_db)
        print(f'  frame {i + 1}/{FRAMES_PER_FREQ} captured for {freq_hz/1e6:.3f} MHz')

    psd_stack = np.vstack(frame_psd)
    avg_psd_db = np.mean(psd_stack, axis=0)
    max_psd_db = np.max(psd_stack, axis=0)

    captures.append((freq_hz, stream_id, freq_axis_hz, avg_psd_db, max_psd_db))

for freq_hz, stream_id, freq_axis_hz, avg_psd_db, max_psd_db in captures:
    plt.figure()
    plt.plot(freq_axis_hz / 1e6, avg_psd_db, linewidth=1.0, label='Average (10 frames)')
    plt.plot(freq_axis_hz / 1e6, max_psd_db, linewidth=1.0, alpha=0.9, label='Persistence Max (10 frames)')
    plt.title(f'FFT around {freq_hz/1e6:.3f} MHz (N={len(freq_axis_hz)})')
    plt.xlabel('Frequency (MHz)')
    plt.ylabel('Magnitude (dBFS-ish)')
    plt.legend()
    plt.show()


In [ ]:
# Cell 7 : Stop IQ streams
for _, stream_id, _, _, _ in captures:
    api_post(f'/streams/{stream_id}/stop', {})
    print('Stopped stream:', stream_id)


In [ ]:
# Cell 8 : Sweep test and plot latest sweep row
sweep_payload = {
    'device_id': DEVICE_ID,
    'start_freq_hz': 902_000_000,
    'stop_freq_hz': 928_000_000,
    'bin_width_hz': 100_000,
    'lna_gain_db': LNA_GAIN_DB,
    'vga_gain_db': VGA_GAIN_DB,
    'amp_enable': AMP_ENABLE
}

sweep_resp = api_post('/sweeps/start', sweep_payload)
SWEEP_ID = sweep_resp['sweep_id']
print('Started sweep:', SWEEP_ID)

time.sleep(2.0)
samples = api_get(f'/sweeps/{SWEEP_ID}/samples')
print('Sweep samples received:', len(samples))

if samples:
    last = samples[-1]
    db_vals = np.array(last['db_values'], dtype=np.float32)
    x = np.linspace(last['hz_low'], last['hz_high'], len(db_vals), endpoint=False) / 1e6

    plt.figure()
    plt.plot(x, db_vals, linewidth=0.9)
    plt.title('Latest sweep row')
    plt.xlabel('Frequency (MHz)')
    plt.ylabel('Power (dB)')
    plt.show()

api_post(f'/sweeps/{SWEEP_ID}/stop', {})
print('Stopped sweep:', SWEEP_ID)